# Breast Ultrasound Segmentation

U-Net segmentation on BUSI: data validation, training, testing, and mask prediction.

The recorded experiments ran on 2026-10-04. BCE test Dice: 0.3895; focal-loss test Dice: 0.4547. Both use one fixed seed and an image-level test split.


In [ ]:
import os
import sys
from pathlib import Path
assert (3, 12) <= sys.version_info[:2] < (3, 14)
project = Path.cwd()
if (project / 'breast-ultrasound-segmentation' / 'src').is_dir():
    project = project / 'breast-ultrasound-segmentation'
elif not (project / 'src').is_dir():
    project = Path('/content/busi') if Path('/content').is_dir() else project / 'busi'
for folder in ['src/busi_segmentation', 'scripts', 'tests', 'configs', 'results', 'runs']:
    (project / folder).mkdir(parents=True, exist_ok=True)
os.chdir(project)
os.environ['MPLCONFIGDIR'] = str(project / '.cache/matplotlib')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'


## Data and model


In [ ]:
%%writefile src/busi_segmentation/__init__.py
"""Reproducible BUSI segmentation experiments."""


In [ ]:
%%writefile src/busi_segmentation/data.py
"""Strict image/mask pairing and persisted, stratified image-level splits."""
from pathlib import Path
import hashlib
import json
import random
import re
import numpy as np
from PIL import Image

CATEGORIES = ('benign', 'malignant', 'normal')


def discover(root, exclusions=None):
    root = Path(root)
    exclusions = exclusions or []
    excluded = {r['id']: r for r in exclusions}
    verified_exclusions = set()
    records, hashes = [], {}
    for category in CATEGORIES:
        folder = root / category
        if not folder.is_dir():
            raise ValueError(f'Missing category directory: {folder}')
        images = sorted(p for p in folder.glob('*.png') if '_mask' not in p.stem)
        if len(images) < 7:
            raise ValueError(f'Need at least 7 images per category: {category}')
        used_masks = set()
        for image in images:
            pattern = re.compile(re.escape(image.stem) + r'_mask(?:_\d+)?\.png$')
            masks = sorted(p for p in folder.glob('*.png') if pattern.fullmatch(p.name))
            if not masks:
                raise ValueError(f'Missing mask for {image}')
            used_masks.update(masks)
            with Image.open(image) as source:
                pixels = np.asarray(source.convert('L'))
            digest = hashlib.sha256(str(pixels.shape).encode() + pixels.tobytes()).hexdigest()
            identity = f'{category}/{image.name}'
            if identity in excluded:
                if digest != excluded[identity]['image_sha256']:
                    raise ValueError(f'Excluded image changed: {identity}')
                verified_exclusions.add(identity)
                continue
            if digest in hashes:
                raise ValueError(f'Duplicate decoded image: {image} and {hashes[digest]}. Resolve before splitting.')
            hashes[digest] = str(image)
            records.append({'id': f'{category}/{image.name}', 'category': category,
                            'image': image.relative_to(root).as_posix(),
                            'masks': [p.relative_to(root).as_posix() for p in masks],
                            'image_sha256': digest,
                            'mask_sha256': [hashlib.sha256(p.read_bytes()).hexdigest() for p in masks]})
        orphan = set(folder.glob('*_mask*.png')) - used_masks
        if orphan:
            raise ValueError(f'Orphan masks in {folder}: {sorted(p.name for p in orphan)}')
    if verified_exclusions != set(excluded):
        raise ValueError('An exclusion refers to an image absent from this dataset')
    return records


def make_splits(records, seed=42):
    rng = random.Random(seed)
    splits = {name: [] for name in ('train', 'val', 'test')}
    for category in CATEGORIES:
        group = sorted((r for r in records if r['category'] == category), key=lambda r: r['id'])
        rng.shuffle(group)
        n_test = max(1, round(len(group) * .15))
        n_val = max(1, round(len(group) * .15))
        splits['test'].extend(group[:n_test])
        splits['val'].extend(group[n_test:n_test+n_val])
        splits['train'].extend(group[n_test+n_val:])
    return {'protocol': 'stratified-image-level-70-15-15-v1', 'seed': seed, 'splits': splits}


def load_sample(root, record, size=128):
    root = Path(root)
    with Image.open(root / record['image']) as source:
        shape = source.size
        image = np.asarray(source.convert('L').resize((size, size), Image.Resampling.BILINEAR), dtype=np.float32) / 255
    union = np.zeros((shape[1], shape[0]), dtype=bool)
    for path in record['masks']:
        with Image.open(root / path) as source:
            if source.size != shape:
                raise ValueError(f'Mask/image dimension mismatch: {path}')
            union |= np.asarray(source.convert('L')) > 127
    if record['category'] == 'normal' and union.any():
        raise ValueError(f'Normal image has foreground annotation: {record["id"]}')
    mask = np.asarray(Image.fromarray(union.astype(np.uint8) * 255).resize((size, size), Image.Resampling.NEAREST)) > 127
    return image[..., None], mask.astype(np.float32)[..., None]


def prepare(root, destination, exclusions=None):
    destination = Path(destination)
    excluded = json.loads(Path(exclusions).read_text()) if exclusions else []
    manifest = make_splits(discover(root, excluded))
    manifest['exclusions'] = excluded
    # Validate every mask before accepting a manifest.
    for records in manifest['splits'].values():
        for record in records:
            load_sample(root, record)
    if destination.exists():
        if json.loads(destination.read_text()) != manifest:
            raise ValueError('Existing manifest differs. Use a new path after investigating the dataset change.')
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(json.dumps(manifest, indent=2) + '\n')
    return manifest


In [ ]:
%%writefile src/busi_segmentation/model.py
"""Configurable U-Net adapted from the original class project."""
import tensorflow as tf


def conv_block(inputs, filters, residual=False, activation='relu'):
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    if residual:
        shortcut = tf.keras.layers.Conv2D(filters, 1, padding='same')(inputs)
        x = tf.keras.layers.Add()([x, shortcut])
    return tf.keras.layers.Activation(activation)(x)


def attention_gate(skip, gate, filters, activation='relu'):
    theta = tf.keras.layers.Conv2D(filters, 1, padding='same')(skip)
    theta = tf.keras.layers.BatchNormalization()(theta)
    phi = tf.keras.layers.Conv2D(filters, 1, padding='same')(gate)
    phi = tf.keras.layers.BatchNormalization()(phi)
    x = tf.keras.layers.Activation(activation)(tf.keras.layers.Add()([theta, phi]))
    coefficients = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(x)
    return tf.keras.layers.Multiply()([skip, coefficients])


def unet_model(input_shape=(128, 128, 1), num_classes=1, attention=False,
               residual=False, activation='relu', base_filters=16):
    """Four encoder/decoder levels; use base_filters=64 for original width."""
    if base_filters < 2 or num_classes != 1:
        raise ValueError('At least two base filters and one binary output are required')
    if len(input_shape) != 3 or any(s is None or s < 16 or s % 16 for s in input_shape[:2]):
        raise ValueError('Spatial dimensions must be positive multiples of 16')
    inputs = tf.keras.Input(shape=input_shape)
    x, skips = inputs, []
    for level in range(4):
        filters = base_filters * 2**level
        x = conv_block(x, filters, residual, activation)
        skips.append(x)
        x = tf.keras.layers.MaxPool2D(2)(x)
    x = conv_block(x, base_filters * 16, residual, activation)
    for level, skip in reversed(list(enumerate(skips))):
        filters = base_filters * 2**level
        x = tf.keras.layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
        if attention:
            skip = attention_gate(skip, x, max(1, filters // 2), activation)
        x = conv_block(tf.keras.layers.Concatenate()([x, skip]), filters, residual, activation)
    outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(x)
    return tf.keras.Model(inputs, outputs, name='U-Net')


def hard_dice_per_image(y_true, y_pred):
    """Return a vector so Keras averages samples rather than batches."""
    truth = tf.cast(y_true, tf.float32)
    pred = tf.cast(y_pred > .5, tf.float32)
    intersection = tf.reduce_sum(truth * pred, axis=(1, 2, 3))
    denominator = tf.reduce_sum(truth + pred, axis=(1, 2, 3))
    return tf.where(denominator > 0, tf.math.divide_no_nan(2 * intersection, denominator), tf.ones_like(denominator))


def dice_loss(y_true, y_pred, smooth=1e-6):
    truth = tf.cast(y_true, y_pred.dtype)
    intersection = tf.reduce_sum(truth * y_pred, axis=(1, 2, 3))
    denominator = tf.reduce_sum(truth + y_pred, axis=(1, 2, 3))
    return 1 - tf.reduce_mean((2 * intersection + smooth) / (denominator + smooth))


def iou_loss(y_true, y_pred, smooth=1e-6):
    truth = tf.cast(y_true, y_pred.dtype)
    intersection = tf.reduce_sum(truth * y_pred, axis=(1, 2, 3))
    union = tf.reduce_sum(truth + y_pred, axis=(1, 2, 3)) - intersection
    return 1 - tf.reduce_mean((intersection + smooth) / (union + smooth))


In [ ]:
%%writefile src/busi_segmentation/metrics.py
"""Per-image foreground scores; empty/empty scores 1, empty/nonempty scores 0."""
import numpy as np


def score_mask(truth, probabilities, threshold=.5):
    truth = np.asarray(truth)
    probabilities = np.asarray(probabilities)
    if truth.shape != probabilities.shape or not truth.size:
        raise ValueError('Masks must have the same, nonempty shape')
    if not np.isfinite(truth).all() or not np.isfinite(probabilities).all() or not 0 <= threshold <= 1:
        raise ValueError('Masks must be finite and threshold in [0,1]')
    if np.any((probabilities < 0) | (probabilities > 1)) or not np.isin(truth, [0, 1]).all():
        raise ValueError('Ground truth must be binary and probabilities in [0,1]')
    truth = truth > .5
    pred = probabilities > threshold
    intersection = int(np.count_nonzero(truth & pred))
    total = int(truth.sum() + pred.sum())
    union = total - intersection
    return {'dice': 2 * intersection / total if total else 1.,
            'iou': intersection / union if union else 1.,
            'pixel_accuracy': float(np.mean(truth == pred)),
            'true_pixels': int(truth.sum()), 'predicted_pixels': int(pred.sum())}


def summarize(rows):
    if not rows:
        raise ValueError('Cannot summarize an empty evaluation')
    def mean(group):
        return {'n': len(group), **{key: float(np.mean([r[key] for r in group])) if group else None
                                  for key in ('dice', 'iou', 'pixel_accuracy')}}
    normal = [r for r in rows if r['category'] == 'normal']
    return {'all_images': mean(rows),
            'tumor_images': mean([r for r in rows if r['category'] != 'normal']),
            'by_category': {c: mean([r for r in rows if r['category'] == c])
                            for c in ('benign', 'malignant', 'normal')},
            'normal_false_positive_image_rate': float(np.mean([r['predicted_pixels'] > 0 for r in normal])) if normal else None}


In [ ]:
%%writefile src/busi_segmentation/cli.py
"""Prepare data, train on train/validation only, and evaluate a frozen run."""
import argparse
import csv
import json
from pathlib import Path
import platform
import hashlib
import subprocess
import sys
import numpy as np
from .data import prepare, load_sample
from .metrics import score_mask, summarize


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')


def arrays(root, records):
    samples = [load_sample(root, record) for record in records]
    return np.stack([s[0] for s in samples]), np.stack([s[1] for s in samples])


def evaluate_rows(model, root, records, batch_size):
    # Collect individual scores; aggregation is independent of batch size.
    rows = []
    for start in range(0, len(records), batch_size):
        chunk = records[start:start + batch_size]
        x, y = arrays(root, chunk)
        predictions = model.predict(x, verbose=0)
        for record, truth, pred in zip(chunk, y, predictions):
            rows.append({'id': record['id'], 'category': record['category'], **score_mask(truth, pred)})
    return rows


def train(args, manifest):
    import tensorflow as tf
    from .model import unet_model, dice_loss, iou_loss, hard_dice_per_image
    run = Path(args.run)
    if run.exists():
        raise ValueError('Run directory already exists. Choose a new run path to preserve evidence.')
    tf.keras.utils.set_random_seed(args.seed)
    tf.config.threading.set_intra_op_parallelism_threads(args.cpu_threads)
    tf.config.threading.set_inter_op_parallelism_threads(2)
    tf.config.experimental.enable_op_determinism()
    config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}
    losses = {'bce': 'binary_crossentropy', 'dice': dice_loss, 'iou': iou_loss,
              'focal': tf.keras.losses.BinaryFocalCrossentropy(apply_class_balancing=True, alpha=args.alpha, gamma=2)}

    def augment(image, mask):
        joined = tf.concat([image, mask], axis=-1)
        joined = tf.image.random_flip_left_right(joined)
        joined = tf.image.random_flip_up_down(joined)
        joined = tf.image.rot90(joined, tf.random.uniform((), maxval=4, dtype=tf.int32))
        return joined[..., :1], joined[..., 1:]

    x, y = arrays(args.data_root, manifest['splits']['train'])
    vx, vy = arrays(args.data_root, manifest['splits']['val'])
    ds = tf.data.Dataset.from_tensor_slices((x, y)).shuffle(len(x), seed=args.seed)
    if not args.no_augmentation:
        # Serial mapping makes the seeded random augmentation order explicit.
        ds = ds.map(augment)
    ds = ds.batch(args.batch_size).prefetch(tf.data.AUTOTUNE)
    val = tf.data.Dataset.from_tensor_slices((vx, vy)).batch(16)
    model = unet_model(input_shape=(128, 128, 1), residual=args.residual,
                       attention=args.attention, activation=args.activation, base_filters=args.base_filters)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=losses[args.loss],
                  metrics=[tf.keras.metrics.MeanMetricWrapper(hard_dice_per_image, name='dice'),
                           tf.keras.metrics.BinaryAccuracy(name='pixel_accuracy')])
    run.mkdir(parents=True)
    write_json(run / 'config.json', config)
    write_json(run / 'manifest.json', manifest)
    write_json(run / 'source_hashes.json', {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                                           for p in Path(__file__).parent.glob('*.py')})
    write_json(run / 'environment.json', {'python': platform.python_version(), 'tensorflow': tf.__version__,
                                         'numpy': np.__version__, 'platform': platform.platform(),
                                         'devices': [str(d) for d in tf.config.list_physical_devices()]})
    (run / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
    checkpoint = run / 'best.weights.h5'
    history = model.fit(ds, validation_data=val, epochs=args.epochs, verbose=2, shuffle=False, callbacks=[
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.ModelCheckpoint(str(checkpoint), monitor='val_dice', mode='max', save_best_only=True, save_weights_only=True),
        tf.keras.callbacks.EarlyStopping(monitor='val_dice', mode='max', patience=15, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_dice', mode='max', patience=5, factor=.5, min_lr=1e-6),
        tf.keras.callbacks.CSVLogger(str(run / 'history.csv'))])
    optimizer_updates = int(model.optimizer.iterations.numpy())
    if any(not np.isfinite(values).all() for values in history.history.values()):
        raise ValueError('Training produced nonfinite values; this run is not valid')
    model.load_weights(checkpoint)
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, metric in zip(axes, ('loss', 'dice')):
        ax.plot(history.history[metric], label='train')
        ax.plot(history.history['val_' + metric], label='validation')
        ax.set_xlabel('Epoch (zero indexed)')
        ax.set_ylabel(metric)
        ax.legend()
    fig.tight_layout()
    fig.savefig(run / 'training_curves.png', dpi=150)
    plt.close(fig)
    write_json(run / 'training_summary.json', {'epochs_completed': len(history.epoch), 'parameters': model.count_params(),
                                               'optimizer_updates': optimizer_updates,
                                               'best_epoch': int(np.argmax(history.history['val_dice'])) + 1,
                                               'checkpoint_metric': 'validation all-image macro Dice'})
    write_json(run / 'validation.json', summarize(evaluate_rows(model, args.data_root, manifest['splits']['val'], 16)))
    print(f'Training complete. Review {run}/validation.json. Test data has not been evaluated.')


def evaluate(args, manifest):
    import tensorflow as tf
    tf.config.threading.set_intra_op_parallelism_threads(args.cpu_threads)
    tf.config.threading.set_inter_op_parallelism_threads(2)
    from .model import unet_model
    run = Path(args.run)
    config = json.loads((run / 'config.json').read_text())
    saved = json.loads((run / 'manifest.json').read_text())
    if manifest != saved:
        raise ValueError('Current data manifest differs from the training manifest')
    if (run / 'test_metrics.json').exists():
        raise ValueError('Test results already exist; preserve them rather than repeatedly probing test performance')
    model = unet_model(input_shape=(128, 128, 1), residual=config['residual'],
                       attention=config['attention'], activation=config['activation'], base_filters=config['base_filters'])
    model.load_weights(run / 'best.weights.h5')
    rows = evaluate_rows(model, args.data_root, manifest['splits']['test'], args.batch_size)
    with (run / 'test_per_image.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    empty = []
    for record in manifest['splits']['test']:
        _, truth = load_sample(args.data_root, record)
        empty.append({'category': record['category'], **score_mask(truth, np.zeros_like(truth))})
    write_json(run / 'test_metrics.json', {'threshold': .5, 'model': summarize(rows), 'all_background_baseline': summarize(empty)})
    # Show both successes and failures among tumor cases, determined by per-image Dice.
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    ranked = sorted((r for r in rows if r['category'] != 'normal'), key=lambda r: r['dice'])
    chosen = ranked[:3] + ranked[-3:]
    lookup = {r['id']: r for r in manifest['splits']['test']}
    fig, axes = plt.subplots(len(chosen), 3, figsize=(9, 3 * len(chosen)), squeeze=False)
    for i, row in enumerate(chosen):
        image, mask = load_sample(args.data_root, lookup[row['id']])
        pred = model.predict(image[None], verbose=0)[0] > .5
        for ax, value, title in zip(axes[i], (image, mask, pred), ('Ultrasound', 'Ground truth', 'Prediction')):
            ax.imshow(value.squeeze(), cmap='gray'); ax.set_title(title); ax.axis('off')
        axes[i, 0].set_title(f'{row["id"]}\nDice={row["dice"]:.3f}', fontsize=8)
    fig.tight_layout(); fig.savefig(run / 'test_examples.png', dpi=150); plt.close(fig)
    print(f'Test evaluation saved in {run}. Use these results for reporting; tune only on validation.')


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('command', choices=['prepare', 'train', 'evaluate'])
    parser.add_argument('--data-root', type=Path, required=True, help='Folder containing benign/, malignant/, normal/')
    parser.add_argument('--manifest', type=Path, default=Path('results/splits.json'))
    parser.add_argument('--exclusions', type=Path, default=Path('configs/data_exclusions.json'), help='Audited BUSI exclusion rules')
    parser.add_argument('--run', type=Path, default=Path('runs/baseline_seed42'))
    parser.add_argument('--seed', type=int, default=42, help='Training seed; split seed is fixed at 42')
    parser.add_argument('--batch-size', type=int, default=16)
    parser.add_argument('--epochs', type=int, default=60)
    parser.add_argument('--base-filters', type=int, default=16, help='16 for compact U-Net; 64 for original width')
    parser.add_argument('--cpu-threads', type=int, default=4)
    parser.add_argument('--loss', choices=['bce', 'focal', 'dice', 'iou'], default='bce')
    parser.add_argument('--alpha', type=float, default=.7)
    parser.add_argument('--activation', choices=['relu', 'swish'], default='relu')
    parser.add_argument('--residual', action='store_true')
    parser.add_argument('--attention', action='store_true')
    parser.add_argument('--no-augmentation', action='store_true')
    args = parser.parse_args()
    if args.batch_size < 1 or args.epochs < 1 or args.base_filters < 2 or args.cpu_threads < 1 or not 0 <= args.alpha <= 1:
        parser.error('Batch size, epochs, and thread count must be positive; base filters >= 2; alpha in [0,1]')
    manifest = prepare(args.data_root, args.manifest, args.exclusions)
    if args.command == 'prepare':
        print({name: len(records) for name, records in manifest['splits'].items()})
    elif args.command == 'train':
        train(args, manifest)
    else:
        evaluate(args, manifest)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile src/busi_segmentation/predict.py
"""Run a saved segmenter on one image and export its mask and overlay."""
import argparse
import json
from pathlib import Path
import numpy as np
from PIL import Image


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--run', type=Path, required=True)
    parser.add_argument('--image', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True, help='New directory for prediction artifacts')
    args = parser.parse_args()
    if args.output.exists():
        parser.error('Output directory exists; choose a new path')
    import tensorflow as tf
    tf.config.threading.set_intra_op_parallelism_threads(4)
    tf.config.threading.set_inter_op_parallelism_threads(2)
    from .model import unet_model
    config = json.loads((args.run / 'config.json').read_text())
    model = unet_model(residual=config['residual'], attention=config['attention'],
                       activation=config['activation'], base_filters=config['base_filters'])
    model.load_weights(args.run / 'best.weights.h5')
    with Image.open(args.image) as source:
        gray = source.convert('L')
        image = np.asarray(gray.resize((128, 128), Image.Resampling.BILINEAR), dtype=np.float32) / 255
        original = np.asarray(gray.convert('RGB'), dtype=np.float32)
    probabilities = model(image[None, ..., None], training=False).numpy()[0, ..., 0]
    mask = Image.fromarray((probabilities > .5).astype(np.uint8) * 255)
    scaled_mask = mask.resize(gray.size, Image.Resampling.NEAREST)
    foreground = np.asarray(scaled_mask) > 0
    overlay = original.copy()
    overlay[foreground] = .6 * overlay[foreground] + .4 * np.array([255, 0, 0])
    args.output.mkdir(parents=True)
    mask.save(args.output / 'mask_128.png')
    scaled_mask.save(args.output / 'mask_original_size.png')
    Image.fromarray(overlay.astype(np.uint8)).save(args.output / 'overlay.png')
    np.save(args.output / 'probabilities.npy', probabilities)
    (args.output / 'prediction.json').write_text(json.dumps({'image': str(args.image), 'run': str(args.run),
        'threshold': .5, 'model_input_size': [128, 128], 'original_size': list(gray.size),
        'foreground_pixels_128': int((probabilities > .5).sum())}, indent=2) + '\n')
    print(args.output)


if __name__ == '__main__':
    main()


## Experiment settings


In [ ]:
%%writefile configs/data_exclusions.json
[
  {
    "id": "benign/benign (433).png",
    "image_sha256": "bea8a9762e7799e6a3db2431e0719580748c65a50aabbc0df760ca3d6bd9c812",
    "reason": "Exact decoded-image duplicate across benign/malignant categories with different foreground masks; exclude both ambiguous records before splitting."
  },
  {
    "id": "malignant/malignant (145).png",
    "image_sha256": "bea8a9762e7799e6a3db2431e0719580748c65a50aabbc0df760ca3d6bd9c812",
    "reason": "Exact decoded-image duplicate across benign/malignant categories with different foreground masks; exclude both ambiguous records before splitting."
  }
]


In [ ]:
%%writefile configs/study.json
{
  "protocol": "corrected-original-v1",
  "dataset": "aryashah2k/breast-ultrasound-images-dataset/versions/1",
  "split": "image-level stratified 70/15/15; seed 42; two audited ambiguous duplicates excluded",
  "training_seeds": [42],
  "epochs": 30,
  "batch_size": 16,
  "base_filters": 16,
  "learning_rate": 0.0001,
  "threshold": 0.5,
  "experiments": [
    {"name": "baseline_bce_seed42", "loss": "bce", "activation": "relu", "residual": false, "attention": false},
    {"name": "focal07_seed42", "loss": "focal", "alpha": 0.7, "gamma": 2, "activation": "relu", "residual": false, "attention": false}
  ],
  "selection": "checkpoint by validation all-image macro Dice; compare both prespecified configurations on test after training completes",
  "interpretation": "Single-seed exploratory comparison; no statistical superiority or patient-level generalization claim. Compact width differs from the original 64-channel model; historical scores are not directly comparable."
}


## Execution scripts


In [ ]:
%%writefile scripts/download_data.py
"""Download a pinned BUSI mirror version into the project, not the user's global cache."""
import json
import os
from pathlib import Path

project = Path(__file__).resolve().parents[1]
os.environ['KAGGLEHUB_CACHE'] = str(project / 'data' / 'cache')
import kagglehub

handle = 'aryashah2k/breast-ultrasound-images-dataset/versions/1'
path = Path(kagglehub.dataset_download(handle))
root = path / 'Dataset_BUSI_with_GT'
if not root.is_dir():
    raise FileNotFoundError(f'Dataset layout differs: {path}')
record = {'handle': handle, 'local_root': str(root), 'kagglehub_version': kagglehub.__version__}
(project / 'data' / 'source.json').write_text(json.dumps(record, indent=2) + '\n')
print(root)


In [ ]:
%%writefile scripts/run_study.py
"""Execute the prespecified corrected study; keep test evaluation separate."""
import json
import os
from pathlib import Path
import subprocess
import sys

project = Path(__file__).resolve().parents[1]
os.chdir(project)
os.environ.setdefault('MPLCONFIGDIR', str(project / '.cache' / 'matplotlib'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
source = json.loads((project / 'data/source.json').read_text())
plan = json.loads((project / 'configs/study.json').read_text())
for experiment in plan['experiments']:
    run = project / 'runs' / experiment['name']
    if (run / 'training_summary.json').exists() and (run / 'validation.json').exists():
        print(f'Preserving completed run: {run}', flush=True)
        continue
    command = [sys.executable, '-m', 'busi_segmentation.cli', 'train', '--data-root', source['local_root'],
               '--run', str(run), '--epochs', str(plan['epochs']), '--seed', str(plan['training_seeds'][0]),
               '--base-filters', str(plan['base_filters']), '--batch-size', str(plan['batch_size']),
               '--loss', experiment['loss'], '--activation', experiment['activation']]
    if experiment.get('alpha') is not None:
        command.extend(['--alpha', str(experiment['alpha'])])
    if experiment['residual']:
        command.append('--residual')
    if experiment['attention']:
        command.append('--attention')
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)
print('Training complete. Review validation logs before the separate final evaluation.', flush=True)


In [ ]:
%%writefile scripts/evaluate_study.py
"""Freeze validation decisions, then evaluate the two prespecified configurations."""
import json
import hashlib
import os
from pathlib import Path
import subprocess
import sys

project = Path(__file__).resolve().parents[1]
os.chdir(project)
os.environ.setdefault('MPLCONFIGDIR', str(project / '.cache' / 'matplotlib'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
plan_path = project / 'configs/study.json'
plan = json.loads(plan_path.read_text())
source = json.loads((project / 'data/source.json').read_text())
lines = ['# Frozen final-evaluation record', '',
         'Both configurations were prespecified in `configs/study.json` before full training.',
         'Checkpoints were selected on validation all-image macro Dice. Threshold remains 0.5.',
         'No configuration is selected or tuned using test scores.', '',
         '| Run | Validation Dice | Tumor validation Dice | Best epoch |',
         '|---|---:|---:|---:|']
for experiment in plan['experiments']:
    run = project / 'runs' / experiment['name']
    config = json.loads((run / 'config.json').read_text())
    summary = json.loads((run / 'training_summary.json').read_text())
    val = json.loads((run / 'validation.json').read_text())
    for key in ['loss', 'activation', 'residual', 'attention']:
        if config[key] != experiment[key]:
            raise ValueError(f'Run differs from the frozen experiment plan: {key}')
    for key in ['epochs', 'batch_size', 'base_filters']:
        if config[key] != plan[key]:
            raise ValueError(f'Run differs from the frozen budget: {key}')
    if config['seed'] != plan['training_seeds'][0]:
        raise ValueError('Training seed differs from the plan')
    lines.append(f"| {experiment['name']} | {val['all_images']['dice']:.6f} | {val['tumor_images']['dice']:.6f} | {summary['best_epoch']} |")
selection = project / 'results/selection.md'
if selection.exists() and selection.read_text() != '\n'.join(lines) + '\n':
    raise ValueError('An existing final-evaluation record differs; do not silently rewrite it')
selection.write_text('\n'.join(lines) + '\n')
(project / 'results/protocol.json').write_text(json.dumps({'study': plan,
    'study_sha256': hashlib.sha256(plan_path.read_bytes()).hexdigest()}, indent=2) + '\n')
for experiment in plan['experiments']:
    run = project / 'runs' / experiment['name']
    if (run / 'test_metrics.json').exists():
        print(f'Preserving prior test results: {run}', flush=True)
        continue
    subprocess.run([sys.executable, '-m', 'busi_segmentation.cli', 'evaluate',
                    '--data-root', source['local_root'], '--run', str(run)], check=True)
subprocess.run([sys.executable, 'scripts/export_results.py'], check=True)


In [ ]:
%%writefile scripts/export_results.py
"""Export reviewed evidence without raw images or trained weights."""
import csv
import json
from pathlib import Path
import shutil

project = Path(__file__).resolve().parents[1]
plan = json.loads((project / 'configs/study.json').read_text())
rows = []
for experiment in plan['experiments']:
    name = experiment['name']
    run = project / 'runs' / name
    metrics = json.loads((run / 'test_metrics.json').read_text())
    val = json.loads((run / 'validation.json').read_text())
    summary = json.loads((run / 'training_summary.json').read_text())
    target = project / 'results' / name
    target.mkdir(exist_ok=True)
    for filename in ['config.json', 'environment.json', 'source_hashes.json', 'pip-freeze.txt',
                     'history.csv', 'training_summary.json', 'validation.json',
                     'test_per_image.csv', 'test_metrics.json', 'training_curves.png', 'verification.json']:
        if filename == 'verification.json' and not (run / filename).exists():
            continue
        if filename == 'config.json':
            config = json.loads((run / filename).read_text())
            for key in ['data_root', 'run', 'manifest', 'exclusions']:
                config[key] = {'data_root': 'data/cache/datasets/aryashah2k/breast-ultrasound-images-dataset/versions/1/Dataset_BUSI_with_GT',
                               'run': 'runs/' + name, 'manifest': 'results/splits.json',
                               'exclusions': 'configs/data_exclusions.json'}[key]
            (target / filename).write_text(json.dumps(config, indent=2) + '\n')
        elif filename == 'pip-freeze.txt':
            lines=(run / filename).read_text().splitlines()
            (target / filename).write_text('\n'.join(line for line in lines if not line.startswith(('-e ', '# Editable'))) + '\n')
        else:
            shutil.copy2(run / filename, target / filename)
    # Dataset-derived example figures stay in runs/ until redistribution terms permit them.
    row = {'experiment': name, 'validation_dice': val['all_images']['dice'],
           'test_dice': metrics['model']['all_images']['dice'], 'test_iou': metrics['model']['all_images']['iou'],
           'tumor_test_dice': metrics['model']['tumor_images']['dice'],
           'tumor_test_iou': metrics['model']['tumor_images']['iou'],
           'pixel_accuracy': metrics['model']['all_images']['pixel_accuracy'],
           'normal_false_positive_image_rate': metrics['model']['normal_false_positive_image_rate'],
           'epochs_completed': summary['epochs_completed'], 'best_epoch': summary['best_epoch'],
           'parameters': summary['parameters'], 'optimizer_updates': summary['optimizer_updates']}
    rows.append(row)
with (project / 'results/comparison.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
background = metrics['all_background_baseline']
(project / 'results/all_background_baseline.json').write_text(json.dumps(background, indent=2) + '\n')
print(json.dumps(rows, indent=2))


In [ ]:
%%writefile scripts/verify_checkpoint.py
"""Verify saved validation scores and inference batch invariance after reloading weights."""
import argparse
import json
from pathlib import Path
import numpy as np
import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)
from busi_segmentation.model import unet_model
from busi_segmentation.cli import evaluate_rows
from busi_segmentation.metrics import summarize

parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument('--run', type=Path, required=True)
parser.add_argument('--data-root', type=Path, required=True)
args = parser.parse_args()
config = json.loads((args.run / 'config.json').read_text())
manifest = json.loads((args.run / 'manifest.json').read_text())
model = unet_model(base_filters=config['base_filters'], activation=config['activation'],
                   residual=config['residual'], attention=config['attention'])
model.load_weights(args.run / 'best.weights.h5')
records = manifest['splits']['val']
a = summarize(evaluate_rows(model, args.data_root, records, 16))
b = summarize(evaluate_rows(model, args.data_root, records, 7))
saved = json.loads((args.run / 'validation.json').read_text())
for group in ['all_images', 'tumor_images']:
    for metric in ['dice', 'iou', 'pixel_accuracy']:
        np.testing.assert_allclose(a[group][metric], saved[group][metric], atol=1e-6, rtol=0)
        np.testing.assert_allclose(a[group][metric], b[group][metric], atol=1e-4, rtol=0)
result = {'checkpoint_reload_matches_saved_validation': True,
          'validation_batch16_matches_batch7': True, 'validation_images': len(records),
          'all_image_dice': a['all_images']['dice'], 'tumor_image_dice': a['tumor_images']['dice']}
(args.run / 'verification.json').write_text(json.dumps(result, indent=2) + '\n')
print(json.dumps(result, indent=2))


In [ ]:
%%writefile scripts/write_findings.py
"""Write the release report and a pixel-free comparison chart from exported metrics."""
import csv
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

project = Path(__file__).resolve().parents[1]
with (project / 'results/comparison.csv').open() as f:
    rows = list(csv.DictReader(f))
bce = json.loads((project / 'results/baseline_bce_seed42/test_metrics.json').read_text())['model']
focal = json.loads((project / 'results/focal07_seed42/test_metrics.json').read_text())['model']
background = json.loads((project / 'results/all_background_baseline.json').read_text())
zero_counts = {}
for name in ['baseline_bce_seed42', 'focal07_seed42']:
    with (project / 'results' / name / 'test_per_image.csv').open() as f:
        per_image = list(csv.DictReader(f))
    zero_counts[name] = sum(float(r['dice']) == 0 for r in per_image if r['category'] != 'normal')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
labels = ['BCE', 'Focal α=0.7', 'All background']
models = [bce, focal, background]
x = np.arange(3)
for offset, key, title, color in [(-.18, 'all_images', 'All 116 test images', '#3467a8'),
                                  (.18, 'tumor_images', '96 tumor test images', '#d97832')]:
    vals = [m[key]['dice'] for m in models]
    bars = axes[0].bar(x + offset, vals, .34, label=title, color=color)
    axes[0].bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
axes[0].set_xticks(x, labels)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Per-image macro Dice')
axes[0].set_title('Final test overlap')
axes[0].legend(loc='upper right', fontsize=9)
vals = [100 * m['normal_false_positive_image_rate'] for m in models]
bars = axes[1].bar(labels, vals, color=['#3467a8', '#d97832', '#6c7785'], width=.55)
axes[1].bar_label(bars, fmt='%.0f%%', padding=3, fontsize=10)
axes[1].set_ylim(0, 110)
axes[1].set_ylabel('Normal images with any predicted foreground (%)')
axes[1].set_title('False positives on 20 normal test images')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=.2)
    ax.set_axisbelow(True)
fig.suptitle('U-Net loss comparison · seed 42 · image-level test split', fontsize=13)
fig.tight_layout()
fig.savefig(project / 'results/comparison.png', dpi=160)
plt.close(fig)
report = """# Findings

Run date: 2026-10-04. Environment: Python 3.12.14, TensorFlow 2.20.0, macOS arm64 CPU.

## Setup

The BUSI version-1 mirror contains 780 images. Excluding two conflicting duplicate records leaves 778 images: 546 train, 116 validation, and 116 test. All 18 secondary annotation files are included in mask unions.

Both runs use the same 1,946,705-parameter U-Net, seed 42, batch size 16, and Adam at 1e-4. Each completed 30 epochs and 1,050 updates. BCE and class-balanced focal loss (alpha 0.7, gamma 2) are the two training objectives. Validation Dice selected epoch 28 for BCE and epoch 29 for focal loss. The threshold is fixed at > 0.5.

## Test results

| Model | Dice | IoU | Tumor Dice | Tumor IoU | Pixel accuracy | Normal FP image rate |
|---|---:|---:|---:|---:|---:|---:|
"""
for label, metrics in zip(labels, models):
    report += f"| {label} | {metrics['all_images']['dice']:.4f} | {metrics['all_images']['iou']:.4f} | {metrics['tumor_images']['dice']:.4f} | {metrics['tumor_images']['iou']:.4f} | {metrics['all_images']['pixel_accuracy']:.4f} | {metrics['normal_false_positive_image_rate']:.0%} |\n"
report += """
Overall scores average 116 images; tumor scores average the 96 benign/malignant images. Empty ground truth and empty prediction score 1. The all-background reference therefore has nonzero overall overlap despite zero tumor localization.

![Test comparison](comparison.png)

## Observations

"""
delta = focal['all_images']['dice'] - bce['all_images']['dice']
report += f"Focal loss increased observed test Dice by {delta:.4f} absolute. The all-background reference achieved {background['all_images']['pixel_accuracy']:.2%} pixel accuracy with zero tumor Dice, demonstrating why pixel accuracy alone is insufficient.\n\n"
report += f"Both trained models predicted foreground on 19 of 20 normal images. Mean normal foreground area was {100*(1-bce['by_category']['normal']['pixel_accuracy']):.2f}% for BCE and {100*(1-focal['by_category']['normal']['pixel_accuracy']):.2f}% for focal loss. Focal loss improved tumor overlap but produced more foreground area on normal images.\n\n"
report += "## Category scores\n\n| Category | Images | BCE Dice | Focal Dice | BCE IoU | Focal IoU |\n|---|---:|---:|---:|---:|---:|\n"
for cat in ['benign', 'malignant', 'normal']:
    a, b = bce['by_category'][cat], focal['by_category'][cat]
    report += f"| {cat.title()} | {a['n']} | {a['dice']:.4f} | {b['dice']:.4f} | {a['iou']:.4f} | {b['iou']:.4f} |\n"
report += f"\nBCE had {zero_counts['baseline_bce_seed42']} zero-overlap tumor cases; focal loss had {zero_counts['focal07_seed42']}. Prediction figures show missed lesions, incorrect localization, disconnected foreground, and border artifacts. They are generated locally under runs/.\n"
report += """
## Limits

This is a single-seed comparison on one dataset. Splits are image-level, patient identities are unavailable, near duplicates may remain, and there is no external validation. Higher overlap in this run does not establish statistical superiority or clinical performance. Normal-image false positives remain a substantial failure mode.

Configurations, histories, per-image scores, environments, and checkpoint-verification results are retained alongside this report.
"""
(project / 'results/FINDINGS.md').write_text(report)
print('Findings and comparison chart saved.')


## Tests and dependencies


In [ ]:
%%writefile tests/test_protocol.py
import tempfile
import unittest
from pathlib import Path
import numpy as np
from PIL import Image
from busi_segmentation.data import discover, make_splits, load_sample, prepare
from busi_segmentation.metrics import score_mask, summarize


class ProtocolTests(unittest.TestCase):
    def fixture(self, root):
        for j, category in enumerate(('benign', 'malignant', 'normal')):
            folder = root / category
            folder.mkdir()
            for i in range(10):
                pixels = np.full((4, 4), i + j * 10, np.uint8)
                Image.fromarray(pixels).save(folder / f'{category} ({i}).png')
                Image.fromarray(np.zeros((4, 4), np.uint8)).save(folder / f'{category} ({i})_mask.png')

    def test_mask_union_and_splits(self):
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder); self.fixture(root)
            mask = np.zeros((4, 4), np.uint8); mask[0, 0] = 255
            Image.fromarray(mask).save(root / 'benign/benign (0)_mask.png')
            mask[:] = 0; mask[3, 3] = 255
            Image.fromarray(mask).save(root / 'benign/benign (0)_mask_1.png')
            records = discover(root)
            r = next(r for r in records if r['id'] == 'benign/benign (0).png')
            image, union = load_sample(root, r, size=4)
            self.assertEqual(union.sum(), 2)
            self.assertEqual(image.shape, (4, 4, 1))
            manifest = make_splits(records)
            self.assertEqual(manifest, make_splits(records))
            ids = [r['id'] for split in manifest['splits'].values() for r in split]
            self.assertEqual(len(ids), len(set(ids)))
            self.assertEqual(set(ids), {r['id'] for r in records})
            for split in manifest['splits'].values():
                self.assertEqual({r['category'] for r in split}, {'benign', 'malignant', 'normal'})
            self.assertEqual(prepare(root, root / 'splits.json'), prepare(root, root / 'splits.json'))
            Image.fromarray(np.ones((4, 4), np.uint8)).save(root / 'benign/benign (0)_mask.png')
            with self.assertRaisesRegex(ValueError, 'manifest differs'):
                prepare(root, root / 'splits.json')

    def test_missing_or_duplicate_data_rejected(self):
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder); self.fixture(root)
            (root / 'benign/benign (0)_mask.png').unlink()
            with self.assertRaisesRegex(ValueError, 'Missing mask'):
                discover(root)
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder); self.fixture(root)
            (root / 'benign/benign (1).png').write_bytes((root / 'benign/benign (0).png').read_bytes())
            with self.assertRaisesRegex(ValueError, 'Duplicate'):
                discover(root)

    def test_overlap_edge_cases(self):
        empty = np.zeros((2, 2)); full = np.ones((2, 2))
        self.assertEqual(score_mask(empty, empty)['dice'], 1)
        self.assertEqual(score_mask(full, empty)['dice'], 0)
        self.assertEqual(score_mask(empty, full)['iou'], 0)
        half = np.array([[1, 1], [0, 0]])
        self.assertAlmostEqual(score_mask(full, half)['dice'], 2/3)
        self.assertAlmostEqual(score_mask(full, half)['iou'], .5)

    def test_macro_aggregation_is_partition_invariant(self):
        rows = [{'category': 'benign', **score_mask(np.ones((2, 2)), np.zeros((2, 2)))} for _ in range(5)]
        rows.append({'category': 'normal', **score_mask(np.zeros((2, 2)), np.zeros((2, 2)))})
        by_batches = [r for chunk in (rows[:4], rows[4:]) for r in chunk]
        summary = summarize(rows)
        self.assertEqual(summary, summarize(by_batches))
        self.assertAlmostEqual(summary['all_images']['dice'], 1/6)
        self.assertEqual(summary['tumor_images']['dice'], 0)

    def test_annotation_errors_rejected(self):
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder); self.fixture(root)
            record = next(r for r in discover(root) if r['category'] == 'normal')
            Image.fromarray(np.ones((4, 4), np.uint8) * 255).save(root / record['masks'][0])
            with self.assertRaisesRegex(ValueError, 'foreground annotation'):
                load_sample(root, record)
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder); self.fixture(root)
            record = discover(root)[0]
            Image.fromarray(np.zeros((3, 3), np.uint8)).save(root / record['masks'][0])
            with self.assertRaisesRegex(ValueError, 'dimension mismatch'):
                load_sample(root, record)
            Image.fromarray(np.zeros((4, 4), np.uint8)).save(root / 'benign/orphan_mask.png')
            with self.assertRaisesRegex(ValueError, 'Orphan masks'):
                discover(root)

    def test_explicit_exclusion_is_hash_checked(self):
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder); self.fixture(root)
            original = discover(root)
            record = original[0]
            excluded = {'id': record['id'], 'image_sha256': record['image_sha256'], 'reason': 'test fixture'}
            included = discover(root, [excluded])
            self.assertEqual(len(included), len(original) - 1)
            self.assertNotIn(record['id'], {r['id'] for r in included})
            excluded['image_sha256'] = 'wrong'
            with self.assertRaisesRegex(ValueError, 'Excluded image changed'):
                discover(root, [excluded])

    def test_prediction_rows_are_batch_invariant(self):
        from busi_segmentation.cli import evaluate_rows
        class AllBackground:
            def predict(self, images, verbose=0):
                return np.zeros_like(images)
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder); self.fixture(root)
            records = discover(root)
            a = evaluate_rows(AllBackground(), root, records, 7)
            b = evaluate_rows(AllBackground(), root, records, 16)
            self.assertEqual(a, b)
            self.assertEqual(len(a), len(records))

    def test_invalid_masks_rejected(self):
        with self.assertRaises(ValueError):
            score_mask(np.ones((2, 2)), np.ones((3, 3)))
        with self.assertRaises(ValueError):
            score_mask(np.ones((2, 2)), np.full((2, 2), float('nan')))
        with self.assertRaises(ValueError):
            score_mask(np.full((2, 2), float('nan')), np.zeros((2, 2)))
        with self.assertRaises(ValueError):
            score_mask(np.ones((2, 2)), np.full((2, 2), 2.))


if __name__ == '__main__':
    unittest.main()


In [ ]:
%%writefile tests/test_model.py
"""Actual TensorFlow checks, skipped only when the training extra is absent."""
import importlib.util
import tempfile
import unittest
from pathlib import Path
import numpy as np

HAS_TF = importlib.util.find_spec('tensorflow') is not None


@unittest.skipUnless(HAS_TF, 'Install the train extra for model tests')
class ModelTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        import tensorflow as tf
        cls.tf = tf
        tf.keras.utils.set_random_seed(42)
        tf.config.threading.set_intra_op_parallelism_threads(4)
        tf.config.threading.set_inter_op_parallelism_threads(2)

    def test_architecture_variants_and_gradients(self):
        from busi_segmentation.model import unet_model, dice_loss, iou_loss
        tf = self.tf
        x = tf.random.uniform((2, 32, 32, 1))
        y = tf.cast(x > .5, tf.float32)
        for options in ({}, {'residual': True}, {'attention': True},
                        {'activation': 'swish'}, {'residual': True, 'attention': True, 'activation': 'swish'}):
            with self.subTest(options=options):
                tf.keras.backend.clear_session()
                model = unet_model(input_shape=(32, 32, 1), base_filters=4, **options)
                with tf.GradientTape() as tape:
                    output = model(x, training=True)
                    loss = dice_loss(y, output) + iou_loss(y, output)
                gradients = tape.gradient(loss, model.trainable_variables)
                self.assertEqual(output.shape, y.shape)
                self.assertTrue(np.isfinite(output.numpy()).all())
                self.assertTrue(all(g is not None and np.isfinite(g.numpy()).all() for g in gradients))
                self.assertTrue(np.any(output.numpy() > 0) and np.all(output.numpy() < 1))

    def test_metric_is_image_weighted(self):
        from busi_segmentation.model import hard_dice_per_image
        metric = self.tf.keras.metrics.MeanMetricWrapper(hard_dice_per_image)
        truth = self.tf.ones((5, 2, 2, 1))
        metric.update_state(truth, self.tf.zeros_like(truth))
        metric.update_state(self.tf.zeros((1, 2, 2, 1)), self.tf.zeros((1, 2, 2, 1)))
        self.assertAlmostEqual(float(metric.result()), 1/6, places=6)

    def test_checkpoint_round_trip(self):
        from busi_segmentation.model import unet_model
        tf = self.tf
        tf.keras.backend.clear_session()
        model = unet_model(input_shape=(32, 32, 1), base_filters=4)
        model.compile(optimizer='adam', loss='binary_crossentropy')
        x = tf.random.uniform((2, 32, 32, 1))
        model.train_on_batch(x, tf.cast(x > .5, tf.float32))
        expected = model(x, training=False).numpy()
        with tempfile.TemporaryDirectory() as folder:
            checkpoint = Path(folder) / 'model.weights.h5'
            model.save_weights(checkpoint)
            restored = unet_model(input_shape=(32, 32, 1), base_filters=4)
            restored.load_weights(checkpoint)
            np.testing.assert_allclose(restored(x, training=False).numpy(), expected, atol=1e-6)

    def test_original_width_forward_pass(self):
        from busi_segmentation.model import unet_model
        self.tf.keras.backend.clear_session()
        model = unet_model(base_filters=64)
        output = model(self.tf.zeros((1, 128, 128, 1)), training=False)
        self.assertEqual(output.shape, (1, 128, 128, 1))
        self.assertTrue(np.isfinite(output.numpy()).all())

    def test_invalid_shapes_rejected(self):
        from busi_segmentation.model import unet_model
        with self.assertRaises(ValueError):
            unet_model(input_shape=(127, 128, 1))
        with self.assertRaises(ValueError):
            unet_model(num_classes=2)


if __name__ == '__main__':
    unittest.main()


In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "busi-segmentation-study"
version = "0.1.0"
description = "An auditable U-Net ablation study of breast ultrasound segmentation"
requires-python = ">=3.12,<3.14"
dependencies = ["numpy>=1.26,<3", "Pillow>=10,<13"]

[project.optional-dependencies]
train = ["tensorflow>=2.16,<2.21", "matplotlib>=3.8,<4"]
data = ["kagglehub>=0.3,<1"]

[tool.setuptools.packages.find]
where = ["src"]


In [ ]:
%%writefile requirements.txt
# Direct dependencies used for the verified local run (Python 3.12, macOS arm64).
numpy==2.5.3
Pillow==12.3.0
tensorflow==2.20.0
keras==3.15.1
matplotlib==3.11.2
kagglehub==0.4.3


## Install and test


In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .
!{sys.executable} -m unittest discover -s tests -v


## Train and evaluate

Validation selects checkpoints. Final test evaluation uses the fixed configurations and threshold.


In [ ]:
!{sys.executable} scripts/download_data.py
!{sys.executable} scripts/run_study.py
!{sys.executable} scripts/evaluate_study.py
!{sys.executable} scripts/write_findings.py


## Results and predictions


In [ ]:
import csv
from IPython.display import display, Image
with Path('results/comparison.csv').open() as f:
    for row in csv.DictReader(f):
        print(row)
for name in ['baseline_bce_seed42', 'focal07_seed42']:
    display(Image(filename=str(Path('runs') / name / 'training_curves.png')))
    display(Image(filename=str(Path('runs') / name / 'test_examples.png')))


Raw data and model checkpoints remain local. In Colab, save runs/ before disconnecting.
